# Транскрибация видео защит ВКР

Runtime: `T4 GPU` или любой GPU. Перед запуском загрузи три подготовленных аудиофайла `IMG_6042.m4a`, `IMG_6044.m4a`, `IMG_6046.m4a`. Ноутбук сначала чистит звук, затем распознаёт `large-v3` с русской подсказкой по терминологии ВКР.

In [ ]:
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install faster-whisper==1.1.1

## Загрузка аудио

В Colab нажми `Choose Files` и выбери локальные файлы из `~/Downloads/defense-audio/`:

- `IMG_6042.m4a`
- `IMG_6044.m4a`
- `IMG_6046.m4a`

In [ ]:
from google.colab import files
uploaded = files.upload()
list(uploaded.keys())

In [ ]:
import json
import subprocess
from pathlib import Path
from faster_whisper import WhisperModel

AUDIO_FILES = [Path('IMG_6042.m4a'), Path('IMG_6044.m4a'), Path('IMG_6046.m4a')]
OUT_DIR = Path('defense_transcripts')
OUT_DIR.mkdir(exist_ok=True)
CLEAN_DIR = Path('clean_audio')
CLEAN_DIR.mkdir(exist_ok=True)

PROMPT = (
    'Защита выпускной квалификационной работы. '
    'Московский Политех. Прикладная информатика. '
    'ВКР, актуальность, цель, задачи, объект, предмет, требования, '
    'функциональные требования, нефункциональные требования, '
    'архитектура, база данных, интерфейс, тестирование, апробация, '
    'экономическая эффективность, комиссия, научный руководитель.'
)

def preprocess_audio(path: Path) -> Path:
    out = CLEAN_DIR / f'{path.stem}.wav'
    # Mono 16 kHz, speech band-pass, mild denoise and loudness normalization.
    subprocess.run([
        'ffmpeg', '-y', '-i', str(path),
        '-vn', '-ac', '1', '-ar', '16000',
        '-af', 'highpass=f=80,lowpass=f=7800,afftdn=nf=-25,loudnorm=I=-16:TP=-1.5:LRA=11',
        str(out),
    ], check=True)
    return out

CLEAN_AUDIO_FILES = [preprocess_audio(path) for path in AUDIO_FILES]

# large-v3 качественнее для русского, medium быстрее. Если GPU не хватает памяти, замени на 'medium'.
MODEL_SIZE = 'large-v3'
model = WhisperModel(MODEL_SIZE, device='cuda', compute_type='float16')

def fmt_ts(seconds: float, sep: str = ',') -> str:
    total_ms = int(round(seconds * 1000))
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f'{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}'

def transcribe_one(path: Path):
    segments_iter, info = model.transcribe(
        str(path),
        language='ru',
        beam_size=8,
        initial_prompt=PROMPT,
        condition_on_previous_text=False,
        temperature=[0.0, 0.2, 0.4],
        vad_filter=True,
        vad_parameters={'min_silence_duration_ms': 500, 'speech_pad_ms': 300},
    )
    segments = []
    for i, s in enumerate(segments_iter, start=1):
        segments.append({'id': i, 'start': s.start, 'end': s.end, 'text': s.text.strip()})
    stem = path.stem
    (OUT_DIR / f'{stem}.txt').write_text(
        '\n'.join(f"[{fmt_ts(s['start'], '.')}] {s['text']}" for s in segments) + '\n',
        encoding='utf-8',
    )
    (OUT_DIR / f'{stem}.srt').write_text(
        '\n\n'.join(f"{s['id']}\n{fmt_ts(s['start'])} --> {fmt_ts(s['end'])}\n{s['text']}" for s in segments) + '\n',
        encoding='utf-8',
    )
    (OUT_DIR / f'{stem}.json').write_text(
        json.dumps({
            'file': str(path),
            'model': MODEL_SIZE,
            'language': info.language,
            'language_probability': info.language_probability,
            'duration': info.duration,
            'segments': segments,
        }, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    return stem, len(segments), info.duration

results = []
for audio in CLEAN_AUDIO_FILES:
    print('Transcribing', audio)
    results.append(transcribe_one(audio))
results

In [ ]:
combined = []
for txt in sorted(OUT_DIR.glob('*.txt')):
    combined.append(f'# {txt.stem}\n\n')
    combined.append(txt.read_text(encoding='utf-8'))
    combined.append('\n')
(OUT_DIR / 'combined_transcript.md').write_text('\n'.join(combined), encoding='utf-8')
!zip -r defense_transcripts.zip defense_transcripts
files.download('defense_transcripts.zip')